In [1]:
import pandas as pd
import numpy as np

# Memuat dataset mentah
df = pd.read_csv("../data/raw/cardio_train.csv", sep=";")

# Membuat salinan untuk proses cleaning
df_clean = df.copy()

# Memeriksa ukuran dataset
print("Ukuran dataset asli:", df.shape)
print("Ukuran salinan dataset:", df_clean.shape)

Ukuran dataset asli: (70000, 13)
Ukuran salinan dataset: (70000, 13)


In [2]:
# Kriteria anomali tekanan darah
invalid_ap_hi = (
    (df_clean["ap_hi"] <= 0) |
    (df_clean["ap_hi"] > 250)
)

invalid_ap_lo = (
    (df_clean["ap_lo"] <= 0) |
    (df_clean["ap_lo"] > 200)
)

invalid_bp_relation = (
    df_clean["ap_hi"] <= df_clean["ap_lo"]
)

# Kriteria anomali tinggi dan berat badan
invalid_height = (
    (df_clean["height"] < 120) |
    (df_clean["height"] > 220)
)

invalid_weight = df_clean["weight"] < 30

# Menggabungkan seluruh kriteria
invalid_all = (
    invalid_ap_hi |
    invalid_ap_lo |
    invalid_bp_relation |
    invalid_height |
    invalid_weight
)

# Menampilkan hasil pemeriksaan
print("Anomali ap_hi:", invalid_ap_hi.sum())
print("Anomali ap_lo:", invalid_ap_lo.sum())
print("Relasi ap_hi <= ap_lo:", invalid_bp_relation.sum())
print("Anomali height:", invalid_height.sum())
print("Anomali weight:", invalid_weight.sum())
print("Total baris unik terindikasi anomali:", invalid_all.sum())
print("Persentase anomali: {:.2f}%".format(
    invalid_all.mean() * 100
))

Anomali ap_hi: 47
Anomali ap_lo: 975
Relasi ap_hi <= ap_lo: 1236
Anomali height: 53
Anomali weight: 7
Total baris unik terindikasi anomali: 1348
Persentase anomali: 1.93%


In [3]:
# Membuat dataset untuk Linear Regression
df_linear = df_clean.loc[~invalid_all].copy()

# Membuat dataset untuk Logistic Regression
df_logistic = df_clean.loc[~invalid_all].copy()

# Memeriksa ukuran dataset setelah cleaning
print("Dataset Linear Regression:", df_linear.shape)
print("Dataset Logistic Regression:", df_logistic.shape)

print("\nJumlah data yang dihapus:")
print("Linear Regression:", len(df_clean) - len(df_linear))
print("Logistic Regression:", len(df_clean) - len(df_logistic))

Dataset Linear Regression: (68652, 13)
Dataset Logistic Regression: (68652, 13)

Jumlah data yang dihapus:
Linear Regression: 1348
Logistic Regression: 1348


In [4]:
# Memeriksa apakah masih ada anomali pada dataset hasil cleaning

print("ap_hi <= 0:", (df_linear["ap_hi"] <= 0).sum())
print("ap_hi > 250:", (df_linear["ap_hi"] > 250).sum())
print("ap_lo <= 0:", (df_linear["ap_lo"] <= 0).sum())
print("ap_lo > 200:", (df_linear["ap_lo"] > 200).sum())
print("ap_hi <= ap_lo:", (df_linear["ap_hi"] <= df_linear["ap_lo"]).sum())
print(
    "Height tidak valid:",
    ((df_linear["height"] < 120) | (df_linear["height"] > 220)).sum()
)
print("Weight tidak valid:", (df_linear["weight"] < 30).sum())

ap_hi <= 0: 0
ap_hi > 250: 0
ap_lo <= 0: 0
ap_lo > 200: 0
ap_hi <= ap_lo: 0
Height tidak valid: 0
Weight tidak valid: 0


In [5]:
# Memisahkan fitur dan target Linear Regression

X_linear = df_linear.drop(columns=["id", "ap_hi", "cardio"])
y_linear = df_linear["ap_hi"]

print("Ukuran X Linear Regression:", X_linear.shape)
print("Ukuran y Linear Regression:", y_linear.shape)
print("\nFitur Linear Regression:")
print(X_linear.columns.tolist())

Ukuran X Linear Regression: (68652, 10)
Ukuran y Linear Regression: (68652,)

Fitur Linear Regression:
['age', 'gender', 'height', 'weight', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']


In [6]:
# Memisahkan fitur dan target Logistic Regression

X_logistic = df_logistic.drop(columns=["id", "cardio"])
y_logistic = df_logistic["cardio"]

print("Ukuran X Logistic Regression:", X_logistic.shape)
print("Ukuran y Logistic Regression:", y_logistic.shape)
print("\nFitur Logistic Regression:")
print(X_logistic.columns.tolist())

Ukuran X Logistic Regression: (68652, 11)
Ukuran y Logistic Regression: (68652,)

Fitur Logistic Regression:
['age', 'gender', 'height', 'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']


In [7]:
from sklearn.model_selection import train_test_split

# Train-test split untuk Linear Regression
X_train_linear, X_test_linear, y_train_linear, y_test_linear = train_test_split(
    X_linear,
    y_linear,
    test_size=0.2,
    random_state=42
)

# Train-test split untuk Logistic Regression
X_train_logistic, X_test_logistic, y_train_logistic, y_test_logistic = train_test_split(
    X_logistic,
    y_logistic,
    test_size=0.2,
    random_state=42,
    stratify=y_logistic
)

# Memeriksa ukuran data
print("LINEAR REGRESSION")
print("X_train:", X_train_linear.shape)
print("X_test:", X_test_linear.shape)
print("y_train:", y_train_linear.shape)
print("y_test:", y_test_linear.shape)

print("\nLOGISTIC REGRESSION")
print("X_train:", X_train_logistic.shape)
print("X_test:", X_test_logistic.shape)
print("y_train:", y_train_logistic.shape)
print("y_test:", y_test_logistic.shape)

LINEAR REGRESSION
X_train: (54921, 10)
X_test: (13731, 10)
y_train: (54921,)
y_test: (13731,)

LOGISTIC REGRESSION
X_train: (54921, 11)
X_test: (13731, 11)
y_train: (54921,)
y_test: (13731,)


In [8]:
# Fitur kategori nominal
categorical_features = ["gender"]

# Fitur numerik Linear Regression
numeric_features_linear = [
    "age", "height", "weight", "ap_lo",
    "cholesterol", "gluc", "smoke", "alco", "active"
]

# Fitur numerik Logistic Regression
numeric_features_logistic = [
    "age", "height", "weight", "ap_hi", "ap_lo",
    "cholesterol", "gluc", "smoke", "alco", "active"
]

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Preprocessing untuk Linear Regression
preprocessor_linear = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features_linear),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Preprocessing untuk Logistic Regression
preprocessor_logistic = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features_logistic),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [10]:
# Linear Regression
X_train_linear_processed = preprocessor_linear.fit_transform(X_train_linear)
X_test_linear_processed = preprocessor_linear.transform(X_test_linear)

# Logistic Regression
X_train_logistic_processed = preprocessor_logistic.fit_transform(X_train_logistic)
X_test_logistic_processed = preprocessor_logistic.transform(X_test_logistic)

# Memeriksa ukuran hasil preprocessing
print("LINEAR REGRESSION")
print("X_train:", X_train_linear_processed.shape)
print("X_test:", X_test_linear_processed.shape)

print("\nLOGISTIC REGRESSION")
print("X_train:", X_train_logistic_processed.shape)
print("X_test:", X_test_logistic_processed.shape)

LINEAR REGRESSION
X_train: (54921, 11)
X_test: (13731, 11)

LOGISTIC REGRESSION
X_train: (54921, 12)
X_test: (13731, 12)


In [11]:
# Menyimpan dataset hasil cleaning
df_cleaned = df_linear.copy()

df_cleaned.to_csv(
    "../data/processed/cardio_cleaned.csv",
    index=False
)

print("Dataset berhasil disimpan!")
print("Ukuran dataset:", df_cleaned.shape)

Dataset berhasil disimpan!
Ukuran dataset: (68652, 13)
